Tablero de experimentos y diagnostico de generalizacion de ResNet50

Proyecto de Investigacion II. Maestria en Inteligencia Artificial, UNI.
Fecha de corte: 9 de noviembre de 2026.

El trabajo reúne los resultados disponibles de la busqueda de hiperparametros, el analisis de errores y calibracion, y las corridas de estabilidad entre semillas. Su finalidad es ordenar la evidencia y facilitar la preparacion del informe parcial.

El procesamiento utiliza archivos de resultados previamente generados. No realiza entrenamiento ni inferencia sobre imagenes y no incorpora TEST ni el grupo 260423 en la seleccion de modelos.

Se consideran tres conjuntos de resultados: ocho ensayos historicos de optimizacion con semilla 42; el analisis de calibracion de FT_D1_PHOTO sobre 127 imagenes de validacion; y seis corridas de ResNet50 correspondientes a conv5 y conv5_block3 con semillas 42, 123 y 2026. Para el analisis por imagen del Sprint 13 se utilizan las predicciones guardadas de las cuatro corridas nuevas.

Las perdidas de entrenamiento y validacion historicas se registraron bajo condiciones distintas. Por ese motivo no se calcula una brecha TRAIN-VALIDATION directa a partir de ellas.

1. Preparacion de archivos

Se identifica la carpeta del repositorio y se comprueba la existencia de los archivos requeridos. Las tablas, graficos y resúmenes que se generen se guardan en 7. resultados/tablero_mlops_sprint13. No se modifican los resultados originales ni los checkpoints.

In [ ]:
from pathlib import Path
import json
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def raiz_proyecto():
    for carpeta in (Path.cwd(), *Path.cwd().parents):
        if (carpeta / "7. resultados").is_dir():
            return carpeta
    raise FileNotFoundError(
        "No se encontró '7. resultados' en la carpeta actual ni en sus padres. "
        "Abre el notebook dentro del repositorio PROYECTO_TI_2."
    )

RAIZ = raiz_proyecto()
RESULTADOS = RAIZ / "7. resultados"
SALIDA = RESULTADOS / "tablero_mlops_sprint13"
SALIDA.mkdir(parents=True, exist_ok=True)

HPO = RESULTADOS / "diagnostico_overfitting" / "resumen_diagnostico_hpo.csv"
PROTOCOLO = RESULTADOS / "diagnostico_overfitting" / "protocolo_diagnostico.json"
RESUMEN_CAL = RESULTADOS / "analisis_errores_calibracion" / "resumen_analisis.md"
BASE_S12 = RESULTADOS / "estabilidad_semillas"
CSV_S12 = BASE_S12 / "comparacion_tres_semillas.csv"
CSV_PARES = BASE_S12 / "diferencias_pareadas_por_semilla.csv"

obligatorios = [HPO, PROTOCOLO, RESUMEN_CAL, CSV_S12, CSV_PARES]
faltantes = [str(p.relative_to(RAIZ)) for p in obligatorios if not p.is_file()]
if faltantes:
    raise FileNotFoundError("Faltan estos archivos del proyecto:\n- " + "\n- ".join(faltantes))

print("Raíz del repositorio:", RAIZ)
print("Carpeta para derivados:", SALIDA)
print("Entradas esenciales verificadas:", len(obligatorios))

2. Registro de la optimizacion historica

Se revisan los ocho ensayos guardados en diagnostico_overfitting. Todos utilizaron la semilla 42 y presentan variaciones en la tasa de aprendizaje y el dropout. Por ello, la comparacion corresponde a una busqueda de configuraciones y no a ocho repeticiones independientes de una misma configuracion.

In [ ]:
hpo = pd.read_csv(HPO).sort_values("trial").reset_index(drop=True)
with PROTOCOLO.open(encoding="utf-8") as f:
    protocolo_hpo = json.load(f)

assert len(hpo) == 8, f"Se esperaban 8 ensayos históricos; hay {len(hpo)}"
assert hpo["semilla"].nunique() == 1, "Las semillas del HPO no son uniformes: revisar comparabilidad"
assert (hpo["aumento_loss_final"] >= -1e-8).all(), "Revisar deterioro negativo"

columnas = ["trial", "learning_rate", "dropout", "epocas", "mejor_epoca_val_loss",
            "min_val_loss", "val_loss_final", "aumento_loss_final", "macro_f1_val",
            "recall_russet_val", "segundos"]
display(hpo[columnas].round(4))

mejor_f1 = hpo.loc[hpo["macro_f1_val"].idxmax()]
mejor_loss = hpo.loc[hpo["min_val_loss"].idxmin()]
print(f"Mejor macro-F1 HPO: trial {int(mejor_f1.trial)} = {mejor_f1.macro_f1_val:.4f}")
print(f"Menor val_loss mínima: trial {int(mejor_loss.trial)} = {mejor_loss.min_val_loss:.4f}")
print(f"Ensayos con aumento de val_loss tras su mínimo: {(hpo.aumento_loss_final > 0).sum()}/{len(hpo)}")
print(f"Tiempo HPO registrado: {hpo.segundos.sum()/60:.1f} minutos (suma de los 8 ensayos)")
print("Advertencia registrada:", protocolo_hpo.get("advertencia", "sin advertencia"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
orden = hpo.sort_values("trial")
ax.bar(orden["trial"].astype(str), orden["aumento_loss_final"], alpha=.75)
ax.set(title="Aumento de val_loss desde el mínimo hasta la última época — HPO histórico",
       xlabel="Ensayo (todos seed 42)", ylabel="Incremento de val_loss")
ax.axhline(0, lw=.8, color="black")
ax.grid(axis="y", alpha=.20)
fig.tight_layout()
fig.savefig(SALIDA / "01_deterioro_val_loss_HPO.png", dpi=170)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.scatter(hpo["min_val_loss"], hpo["macro_f1_val"], s=70)
for _, fila in hpo.iterrows():
    ax.annotate(str(int(fila.trial)), (fila.min_val_loss, fila.macro_f1_val), xytext=(5, 5), textcoords="offset points")
ax.set(xlabel="Mejor val_loss (menor es mejor)", ylabel="Macro-F1 de validación",
       title="HPO: mejor pérdida y macro-F1 no siempre seleccionan el mismo ensayo")
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(SALIDA / "02_metricas_HPO.png", dpi=170)
plt.show()

Interpretacion del diagnostico historico

En los ocho ensayos aumenta la perdida de validacion desde su valor minimo hasta la ultima epoca registrada. El resultado es compatible con sobreajuste, pero no demuestra por si solo una brecha entre entrenamiento y validacion. Para estimarla de manera comparable se requiere evaluar los checkpoints en modo inferencia con el mismo preprocesamiento, sin aumentos estocasticos ni dropout activo y con una funcion de perdida equivalente.

3. Estabilidad entre semillas. Sprint 12

Las seis corridas corresponden a dos modos de ajuste fino, conv5 y conv5_block3, y tres semillas aleatorias: 42, 123 y 2026. Se utilizan las 127 imagenes de validation_original. La media y la desviacion estandar describen la sensibilidad a la semilla dentro del mismo conjunto de validacion. No equivalen a validacion cruzada ni evaluacion externa.

In [ ]:
s12 = pd.read_csv(CSV_S12).copy()
s12["modo"] = s12["modo"].astype(str)
s12["semilla"] = s12["semilla"].astype(int)
assert len(s12) == 6, f"Se esperaban 6 corridas y se recibieron {len(s12)}"
assert s12.groupby("modo")["semilla"].nunique().eq(3).all(), "Se necesitan 3 semillas por modo"
assert set(s12["n"]) == {127}, "Las corridas usan tamaños de validación diferentes"

metricas = ["accuracy", "macro_f1", "precision_russet", "recall_russet",
            "average_precision_russet", "brier_russet"]
resumen_s12 = s12.groupby("modo")[metricas].agg(["mean", "std"])
resumen_s12.columns = [f"{metrica}_{estadistico}" for metrica, estadistico in resumen_s12.columns]
resumen_s12 = resumen_s12.reset_index()

display(s12[["modo", "semilla", "origen", *metricas]].round(4))
display(resumen_s12.round(4))

s12.to_csv(SALIDA / "corridas_Sprint12_normalizadas.csv", index=False)
resumen_s12.to_csv(SALIDA / "resumen_Sprint12_calculado.csv", index=False)

pares_originales = pd.read_csv(CSV_PARES).sort_values("semilla")
pares_calculados = (s12.pivot(index="semilla", columns="modo", values=metricas))
pares_calculados = pd.DataFrame({
    metrica: pares_calculados[(metrica, "conv5_block3")] - pares_calculados[(metrica, "conv5")]
    for metrica in metricas
}).reset_index()
for metrica in metricas:
    if not np.allclose(
        pares_originales[metrica].to_numpy(),
        pares_calculados[metrica].to_numpy(),
        atol=1e-7,
    ):
        raise ValueError(f"Diferencia pareada no coincide para {metrica}")
print("Diferencias pareadas: coinciden con el CSV original")
display(pares_calculados.round(4))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.8, 4.5))
for modo, datos in s12.groupby("modo"):
    datos = datos.sort_values("semilla")
    axs[0].plot(datos.semilla.astype(str), datos.macro_f1, marker="o", label=modo)
    axs[1].plot(datos.semilla.astype(str), datos.recall_russet, marker="o", label=modo)
axs[0].set(ylabel="Macro-F1", xlabel="Semilla", title="Macro-F1 por semilla")
axs[1].set(ylabel="Recall RUSSET", xlabel="Semilla", title="Recall RUSSET por semilla")
for ax in axs:
    ax.grid(alpha=.25)
    ax.legend(loc="best")
fig.suptitle("ResNet50 — comparación pareada en validation_original (n=127)")
fig.tight_layout()
fig.savefig(SALIDA / "03_estabilidad_Sprint12.png", dpi=180)
plt.show()

Interpretacion de los resultados del Sprint 12

El modo conv5_block3 presenta mayor recall RUSSET promedio y un macro-F1 medio ligeramente superior. Conv5 presenta menor dispersion de macro-F1, mayor precision RUSSET y menor Brier promedio.

Las diferencias pareadas cambian de signo entre semillas, de modo que los resultados no establecen una superioridad consistente de una configuracion en todas las metricas. La seleccion definitiva permanece pendiente. TEST y el grupo 260423 se mantienen fuera de esta decision.

4. Calibracion y analisis historico de errores

El notebook 07 analiza la configuracion FT_D1_PHOTO con 127 registros de validacion. La version historica y la recalculada difieren en una sola prediccion. Se revisan por separado porque sus resultados no corresponden a las seis corridas del Sprint 12. Los archivos existentes incluyen curva de precision-recall, calibracion, sensibilidad al umbral y paneles de errores.

In [ ]:
print(RESUMEN_CAL.read_text(encoding="utf-8"))
carpeta_cal = RESULTADOS / "analisis_errores_calibracion"
archivos_cal = sorted(p.name for p in carpeta_cal.iterdir() if p.is_file())
print("Artefactos existentes en el diagnóstico de calibración:")
for nombre in archivos_cal:
    print(" -", nombre)

# Las cifras aquí son únicamente un resumen documental del archivo histórico,
# no resultados recalculados sobre las corridas del Sprint 12.
comparacion_cal = pd.DataFrame([
    {"version": "historico", "n": 127, "macro_f1": .7876, "recall_russet": .7143,
     "precision_russet": .6000, "ap_russet": .7172, "brier_russet": .0984,
     "fn_russet": 6, "fp_russet": 10},
    {"version": "recalculado", "n": 127, "macro_f1": .7781, "recall_russet": .7143,
     "precision_russet": .5769, "ap_russet": .7184, "brier_russet": .0988,
     "fn_russet": 6, "fp_russet": 11},
])
display(comparacion_cal)
comparacion_cal.to_csv(SALIDA / "resumen_calibracion_historica_documental.csv", index=False)

5. Errores recurrentes en las cuatro corridas nuevas

Se verifica que las corridas de semillas 123 y 2026 utilicen las mismas imagenes y etiquetas. Para cada archivo se cuenta en cuantas corridas la prediccion fue incorrecta. Como las evaluaciones usan el mismo conjunto de validacion, los errores observados no son independientes.

La repeticion de errores identifica casos que requieren revision, pero no determina su causa. Las hipotesis sobre textura, iluminacion, etiquetas o condiciones de captura necesitan inspeccion visual y verificacion del origen de cada imagen.

In [ ]:
corridas = {
    "conv5_seed123": "RN50_CONV5_s123_01",
    "conv5_block3_seed123": "RN50_CONV5B3_s123_01",
    "conv5_seed2026": "RN50_CONV5_s2026_01",
    "conv5_block3_seed2026": "RN50_CONV5B3_s2026_01",
}

predicciones = []
for nombre, sub in corridas.items():
    ruta = BASE_S12 / sub / "predicciones_validation_original.csv"
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta evidencia de predicciones: {ruta}")
    t = pd.read_csv(ruta).copy()
    columnas_necesarias = {"archivo", "clase", "prediccion", "probabilidad_RUSSET"}
    assert columnas_necesarias.issubset(t.columns), f"Columnas ausentes en {nombre}"
    assert len(t) == 127 and t.archivo.is_unique, f"Predicciones inválidas en {nombre}"
    t["corrida"] = nombre
    predicciones.append(t)

pred = pd.concat(predicciones, ignore_index=True)
revision = pred.groupby("archivo").agg(
    clase=("clase", "first"),
    n_etiquetas=("clase", "nunique"),
    n_corridas=("corrida", "nunique"),
).reset_index()
assert revision.n_etiquetas.eq(1).all(), "Las etiquetas no coinciden entre corridas"
assert revision.n_corridas.eq(4).all(), "Las corridas no contienen el mismo conjunto"
pred["error"] = pred["clase"] != pred["prediccion"]
conteo = pred.groupby("archivo")["error"].sum().rename("errores_4").reset_index()
revision = revision.merge(conteo, on="archivo")
revision["fecha_nombre"] = revision["archivo"].str.extract(r"-(\d{6})-\d{6}-", expand=False)
revision = revision.sort_values(["clase", "errores_4", "archivo"], ascending=[True, False, True])

tabla_errores = pd.crosstab(revision.clase, revision.errores_4)
display(tabla_errores)
display(revision.loc[revision.errores_4 == 4, ["clase", "archivo", "fecha_nombre"]])
print("Errores en 4/4 corridas:", revision.loc[revision.errores_4.eq(4)].groupby("clase").size().to_dict())
revision.to_csv(SALIDA / "errores_recurrentes_por_imagen.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.7))
conteos = (revision.groupby(["clase", "errores_4"]).size()
                   .unstack(fill_value=0).reindex(columns=range(5), fill_value=0))
conteos.T.plot(kind="bar", ax=ax, width=.78)
ax.set(title="Repetición de errores en cuatro corridas nuevas",
       xlabel="Número de corridas que fallaron en la misma imagen", ylabel="Número de imágenes")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=.22)
fig.tight_layout()
fig.savefig(SALIDA / "04_errores_recurrentes_Sprint13.png", dpi=180)
plt.show()

6. Auditoria de procedencia y fecha de captura

Los nombres de los archivos contienen una fecha nominal. En la validacion registrada, RUSSET aparece en la fecha 250425 y PICADURA en fechas de mayo de 2026. La asociacion entre clase y fecha constituye un posible factor de confusion. Por si sola no demuestra que el modelo utilice el fondo o las condiciones de captura para clasificar.

El grupo 260423 se conserva como conjunto reservado y no se generan predicciones sobre sus imagenes.

In [ ]:
manifiesto = BASE_S12 / "RN50_CONV5_s123_01" / "manifiesto_260423_usado.csv"
if not manifiesto.is_file():
    raise FileNotFoundError(manifiesto)
mm = pd.read_csv(manifiesto)
mm["fecha_nombre"] = mm["archivo"].str.extract(r"-(\d{6})-\d{6}-", expand=False).fillna("sin_fecha")
resumen_particiones = (mm.groupby(["conjunto", "clase"]).size()
                          .rename("n").reset_index())
conteo_fechas = (mm.groupby(["conjunto", "clase", "fecha_nombre"]).size()
                   .rename("n").reset_index())
display(resumen_particiones)
display(conteo_fechas.loc[conteo_fechas.conjunto == "validation_original"])
assert mm.groupby("conjunto").size().get("evaluacion_exploratoria_260423", 0) == 43
conteo_fechas.to_csv(SALIDA / "auditoria_clase_fecha_por_particion.csv", index=False)
print("Advertencia metodológica: las clases de validación no comparten fechas de captura")

7. Tablero consolidado de experimentos

El tablero incluye los ocho ensayos historicos de optimizacion y las seis corridas del Sprint 12. Se registran la configuracion, la semilla, la procedencia, las metricas disponibles y el tiempo cuando existe un valor documentado. Las series se mantienen identificadas por separado porque responden a protocolos experimentales diferentes. No se asignan tiempos ni brechas de generalizacion que no esten registrados.

In [ ]:
# Registros normalizados: a cada grupo de experiencias se le asigna su procedencia.
h = pd.DataFrame({
    "grupo": "HPO histórico", "run_id": hpo.trial.map(lambda x: f"HPO_trial_{x:02d}"),
    "configuracion": "FT_D1_PHOTO (HPO)", "semilla": hpo.semilla, "dataset_eval": "validation (histórico)",
    "macro_f1": hpo.macro_f1_val, "recall_russet": hpo.recall_russet_val,
    "precision_russet": hpo.precision_russet_val,
    "accuracy": hpo.accuracy_val_final, "tiempo_s": hpo.segundos,
    "learning_rate": hpo.learning_rate, "dropout": hpo.dropout,
    "origen": "histórico", "nota": "métrica histórica; mismo seed 42; gap train-val no comparable",
})
s = pd.DataFrame({
    "grupo": "Sprint 12", "run_id": s12.apply(lambda x: f"RN50_{x.modo}_s{int(x.semilla)}", axis=1),
    "configuracion": s12.modo, "semilla": s12.semilla, "dataset_eval": "validation_original (n=127)",
    "macro_f1": s12.macro_f1, "recall_russet": s12.recall_russet,
    "precision_russet": s12.precision_russet, "accuracy": s12.accuracy,
    "tiempo_s": np.nan, "learning_rate": np.nan, "dropout": np.nan,
    "origen": s12.origen, "nota": "3 semillas por modo; no hay tiempo en CSV comparativo",
})
tablero = pd.concat([h, s], ignore_index=True)
tablero.to_csv(SALIDA / "tablero_mlops_consolidado.csv", index=False)
display(tablero)
print("Tablero generado:", len(tablero), "registros (8 HPO + 6 Sprint 12)")

In [ ]:
media_c5 = resumen_s12.loc[resumen_s12.modo.eq("conv5")].iloc[0]
media_b3 = resumen_s12.loc[resumen_s12.modo.eq("conv5_block3")].iloc[0]
fallos = revision.loc[revision.errores_4.eq(4)].groupby("clase").size().to_dict()

texto = f"""Sintesis de resultados. Proyecto de Investigacion II
Fecha de corte: 9 de noviembre de 2026

Historico de optimizacion de hiperparametros

Se revisaron {len(hpo)} ensayos con semilla {int(hpo.semilla.iloc[0])}. Son configuraciones diferentes y no corresponden a validacion cruzada.
La mejor puntuacion macro-F1 se obtuvo en el ensayo {int(mejor_f1.trial)}: {mejor_f1.macro_f1_val:.4f}.
La menor val_loss registrada corresponde al ensayo {int(mejor_loss.trial)}: {mejor_loss.min_val_loss:.4f}.
En {int((hpo.aumento_loss_final > 0).sum())} de {len(hpo)} ensayos, val_loss aumento desde su minimo hasta el ultimo registro.
El tiempo total registrado para los ocho ensayos es {hpo.segundos.sum():.1f} segundos.
La brecha entre TRAIN y VALIDATION bajo condiciones de inferencia equivalentes todavia no fue cuantificada.

Estabilidad del modelo. Sprint 12

Conv5: macro-F1 = {media_c5.macro_f1_mean:.4f} +/- {media_c5.macro_f1_std:.4f}; recall RUSSET = {media_c5.recall_russet_mean:.4f} +/- {media_c5.recall_russet_std:.4f}.
Conv5_block3: macro-F1 = {media_b3.macro_f1_mean:.4f} +/- {media_b3.macro_f1_std:.4f}; recall RUSSET = {media_b3.recall_russet_mean:.4f} +/- {media_b3.recall_russet_std:.4f}.
Conv5_block3 obtuvo mayor recall RUSSET promedio, pero tambien mayor dispersion en macro-F1. Las tres semillas no permiten establecer una ventaja concluyente para todos los criterios.
Las medias y desviaciones se calcularon con el mismo conjunto de validacion y no corresponden a lotes independientes.

Calibracion historica

En FT_D1_PHOTO, el macro-F1 historico fue 0.7876 y el recalculado 0.7781.
El recall RUSSET fue 0.7143 en ambas versiones y el Brier fue 0.0984 y 0.0988, respectivamente.
Se observo una prediccion diferente sobre 127 imagenes. Estos resultados no justifican por si solos cambiar el umbral.

Errores recurrentes del Sprint 13

Se registraron {fallos.get('RUSSET', 0)} imagenes RUSSET y {fallos.get('PICADURA', 0)} imagenes PICADURA con clasificacion incorrecta en las cuatro corridas nuevas.
La asociacion observada entre fecha nominal y clase en validation_original requiere una revision de la procedencia y del protocolo de particion.
El grupo 260423 permanece reservado y no fue utilizado para calcular las predicciones de este analisis.

Actividades pendientes

1. Evaluar TRAIN y VALIDATION de manera equivalente en modo inferencia para medir la brecha real.
2. Inspeccionar visualmente las imagenes con errores persistentes y verificar las etiquetas.
3. Comprobar la comparabilidad del entrenamiento historico de semilla 42 con las corridas nuevas.
4. Comparar una intervencion de regularizacion de manera controlada cuando el protocolo lo justifique.
5. Registrar la latencia de inferencia si se requiere para evaluar la aplicacion operativa.

Registro de version

Rama: sprint-12-estabilidad-semillas.
Commit de referencia: b054265.
El tablero se genera localmente a partir de archivos existentes y no modifica Git.
"""

ruta_sintesis = SALIDA / "sintesis_avance_MLOps.txt"
ruta_sintesis.write_text(texto, encoding="utf-8")
print(texto)
print("Archivos guardados en:", SALIDA)
for archivo in sorted(SALIDA.iterdir()):
    print(" -", archivo.name)


8. Estado del avance y tareas pendientes

Los experimentos cuentan con registro de parametros, semillas, metricas y artefactos. El Sprint 12 permite calcular media y desviacion estandar para dos configuraciones de ResNet50. El historico HPO contiene ocho ensayos y la evidencia de aumento de val_loss. El notebook 07 documenta calibracion y sensibilidad al umbral para una configuracion historica.

Siguen pendientes la cuantificacion de la brecha TRAIN-VALIDATION bajo condiciones equivalentes, la comparacion controlada de una unica estrategia de mitigacion, la revision visual de los errores persistentes y la verificacion de las condiciones de captura y de las etiquetas. Tambien debe comprobarse la comparabilidad exacta de las corridas historicas de semilla 42 y registrar tiempos de inferencia si se incluyen en el informe.

La evaluacion con TEST y con el grupo 260423 queda reservada para una etapa posterior, una vez fijados los criterios de seleccion. La evidencia actual corresponde al desarrollo del modelo y sustenta un informe parcial.